# Metadata Filtering in Vector Search

Without filtering, every query searches the entire vector store. When your knowledge base spans multiple products, departments, or time periods, an unfiltered search returns chunks from the wrong context. The `filter=` parameter on `aquery()` and `astream()` lets you scope retrieval to exactly the documents that matter for a given question.

**What you'll build:** A knowledge base with documents from multiple sources and categories, queried with precise metadata filters.

**Time:** ~10 min | **Difficulty:** Intermediate

Guide: [Metadata Filtering in Vector Search](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/metadata-filtering)

## Prerequisites & Setup

You will need:
- An **OpenAI API key** — set `OPENAI_API_KEY` in the env cell below
- `synapsekit` and `chromadb` installed (run the install cell)

**What you'll learn:**
- How to attach metadata to documents at ingestion time
- How to filter by a single field (`source`, `category`, `department`)
- How to combine multiple filter conditions with `$and` and `$or`
- How to filter by date range
- How metadata filtering interacts with similarity search

In [ ]:
!pip install synapsekit chromadb -q

In [ ]:
import os

# Set your OpenAI API key before running the cells below
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Ingest Documents with Rich Metadata

Metadata is stored alongside the vector and does not affect similarity scores. Rich metadata at ingestion time is what makes filtering possible at query time — you cannot filter on fields you did not set when ingesting.

In [ ]:
from synapsekit.schema import Document
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.chroma import ChromaVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = ChromaVectorStore(
    collection_name="company-kb",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

# Metadata is stored alongside the vector and does not affect similarity scores.
# Rich metadata at ingestion time is what makes filtering possible at query time.
documents = [
    Document(
        page_content="Product Alpha supports single sign-on via SAML 2.0.",
        metadata={"source": "product-alpha-docs.pdf", "category": "security", "product": "alpha", "year": 2024},
    ),
    Document(
        page_content="Product Beta uses OAuth 2.0 for all API authentication.",
        metadata={"source": "product-beta-docs.pdf", "category": "security", "product": "beta", "year": 2024},
    ),
    Document(
        page_content="Product Alpha pricing starts at $49/month per user.",
        metadata={"source": "pricing-2024.pdf", "category": "pricing", "product": "alpha", "year": 2024},
    ),
    Document(
        page_content="Product Beta pricing starts at $99/month per user.",
        metadata={"source": "pricing-2024.pdf", "category": "pricing", "product": "beta", "year": 2024},
    ),
    Document(
        page_content="The 2023 security audit found no critical vulnerabilities in Product Alpha.",
        metadata={"source": "audit-2023.pdf", "category": "security", "product": "alpha", "year": 2023},
    ),
    Document(
        page_content="HR policy: all employees must complete security training annually.",
        metadata={"source": "hr-policy.pdf", "category": "hr", "department": "hr", "year": 2024},
    ),
]

await vectorstore.aadd_documents(documents)
print("Ingested 6 documents with metadata.")

## Step 2: Build the RAG Pipeline

The pipeline is identical to other guides — filtering is applied at query time via the `filter=` parameter.

In [ ]:
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM

rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=embeddings,
    vectorstore=vectorstore,
)

## Step 3: Filter by a Single Field

Without a filter this query would return chunks from both products. Filtering on `product="alpha"` guarantees only Alpha-specific content is retrieved, preventing the LLM from mixing up the two products.

In [ ]:
# Without a filter this query would return chunks from both products.
# Filtering on product="alpha" guarantees only Alpha-specific content is
# retrieved, preventing the LLM from mixing up the two products.
answer = await rag.aquery(
    "What authentication method is supported?",
    filter={"product": "alpha"},
)
print("Alpha only:", answer)

answer = await rag.aquery(
    "What authentication method is supported?",
    filter={"product": "beta"},
)
print("Beta only:", answer)

## Step 4: Filter by Category

Scoping to `category="pricing"` prevents security documents from surfacing in a pricing question even if they contain relevant-sounding words.

In [ ]:
# Scoping to category="pricing" prevents security documents from surfacing
# in a pricing question even if they contain relevant-sounding words.
answer = await rag.aquery(
    "How much does the product cost?",
    filter={"category": "pricing"},
)
print("Pricing answer:", answer)

## Step 5: Combine Conditions with $and

`$and` requires ALL conditions to match. This narrows retrieval to exactly the Alpha security documents from 2024 — the current year only, not the older 2023 audit report.

In [ ]:
# $and requires ALL conditions to match. This narrows retrieval to
# exactly the Alpha security documents from 2024 — the current year only,
# not the older 2023 audit report.
answer = await rag.aquery(
    "Is the product secure?",
    filter={"$and": [{"product": "alpha"}, {"category": "security"}, {"year": 2024}]},
)
print("Alpha security 2024:", answer)

## Step 6: Combine Conditions with $or

`$or` matches if ANY condition is true. Useful for queries that span multiple products when a user has access to more than one.

In [ ]:
# $or matches if ANY condition is true. Useful for queries that span
# multiple products when a user has access to more than one.
answer = await rag.aquery(
    "What are the authentication options?",
    filter={"$or": [{"product": "alpha"}, {"product": "beta"}]},
)
print("Both products:", answer)

## Step 7: Filter by Date Range

Numeric comparisons use `$gte` / `$lte` operators so you can scope queries to a time window. This is essential for "current year policy" questions where an old document would give a stale answer.

In [ ]:
# Numeric comparisons use $gte / $lte operators so you can scope queries
# to a time window. This is essential for "current year policy" questions
# where an old document would give a stale answer.
answer = await rag.aquery(
    "What does the security audit say?",
    filter={"year": {"$gte": 2024}},
)
print("2024+ only:", answer)

## Step 8: Combine Filtering with Streaming

`filter=` works identically on `astream()`. Streaming and filtering are independent features that compose naturally.

In [ ]:
# filter= works identically on astream(). Streaming and filtering are
# independent features that compose naturally.
print("Streaming with filter: ", end="")
async for token in rag.astream(
    "Explain the authentication approach.",
    filter={"product": "beta"},
):
    print(token, end="", flush=True)
print()

## Complete Working Example

A single self-contained cell that ingests documents with metadata and demonstrates single-field, combined, and date-range filters.

In [ ]:
import asyncio
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.chroma import ChromaVectorStore
from synapsekit.schema import Document

async def main():
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = ChromaVectorStore(
        collection_name="company-kb",
        embedding_function=embeddings,
        persist_directory="./chroma_db",
    )

    documents = [
        Document(
            page_content="Product Alpha supports single sign-on via SAML 2.0.",
            metadata={"product": "alpha", "category": "security", "year": 2024},
        ),
        Document(
            page_content="Product Beta uses OAuth 2.0 for all API authentication.",
            metadata={"product": "beta", "category": "security", "year": 2024},
        ),
        Document(
            page_content="Product Alpha pricing starts at $49/month per user.",
            metadata={"product": "alpha", "category": "pricing", "year": 2024},
        ),
        Document(
            page_content="Product Beta pricing starts at $99/month per user.",
            metadata={"product": "beta", "category": "pricing", "year": 2024},
        ),
    ]
    await vectorstore.aadd_documents(documents)

    rag = RAGPipeline(
        llm=OpenAILLM(model="gpt-4o-mini"),
        embeddings=embeddings,
        vectorstore=vectorstore,
    )

    # Single-field filter
    answer = await rag.aquery(
        "What authentication method is used?",
        filter={"product": "alpha"},
    )
    print("Alpha auth:", answer)

    # Combined filter
    answer = await rag.aquery(
        "How much does it cost?",
        filter={"$and": [{"product": "beta"}, {"category": "pricing"}]},
    )
    print("Beta pricing:", answer)

    # Date range filter
    answer = await rag.aquery(
        "What is the current pricing?",
        filter={"year": {"$gte": 2024}},
    )
    print("2024+ pricing:", answer)

asyncio.run(main())

## Next Steps

- [Hybrid BM25 + Vector Search](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/hybrid-bm25-vector) — combine keyword and semantic search within a filtered scope
- [Build a PDF Knowledge Base](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/pdf-knowledge-base) — set up a Chroma collection to filter against
- [VectorStore reference](https://synapsekit.github.io/synapsekit-docs/docs/rag/vector-stores) — full filter operator documentation